In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
import os

In [ ]:
# Read all CSV files from the different_errors directory
def load_experimental_data(directory):
    all_data = []
    
    # Create a Path object for the directory
    dir_path = Path(directory)
    
    # Iterate through all CSV files in the directory
    for file_path in dir_path.glob('*.csv'):
        df = pd.read_csv(file_path)
        df['run'] = int(file_path.stem.split('_')[-1].replace('run', ''))
        all_data.append(df)
    
    # Combine all dataframes
    return pd.concat(all_data, ignore_index=True)

In [ ]:
# Load the data
# directory='taxi/different_errors/'
directory='synth10/different_errors/'


# Define the custom color palette 
# colors = ['black', 'red', 'green', 'blue'] 
# colors = ['black', '#253494', '#2c7fb8', '#41b6c4', '#a1dab4', 'red'] 
colors = ['black', '#00008B', '#0000FF', '#2ca02c', '#ff7f0e', '#d62728']



In [ ]:
# Function to plot the aggregated results
def plot_aggregated_error_per_column(directory, y_col, include_initialization=False):
    # Load the data
    df = load_experimental_data(directory)

    # Group by errorBound and query number, then calculate mean and std
    agg_df = df.groupby(['errorBound', 'i']).agg(
        mean_time=(y_col, 'mean'),
        std_time=(y_col, 'std')
    ).reset_index()

    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]


    color_palette = {error_bound: colors[i % len(colors)] for i, error_bound in enumerate(agg_df['errorBound'].unique())}

    # Create query index for x-axis (assuming 'i' column represents query sequence)
    plt.figure(figsize=(12, 5))
    fontsize=14
    # Plot a line for each error bound without markers
    for error_bound in agg_df['errorBound'].unique():
        data = agg_df[agg_df['errorBound'] == error_bound]
        label = 'Exact' if error_bound == 0 else f'{error_bound * 100:.0f}%'
        plotted_data = data if include_initialization else data[1:]
        plt.plot(plotted_data['i'], plotted_data['mean_time'], 
                 label=label, 
                 linewidth=2,
                 color=color_palette[error_bound])

    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel(y_col, fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize = fontsize, frameon=False)  # Remove the box around the legend
    
    # Remove the grid lines behind the plot
    plt.grid(False)

    # Rotate x-axis labels if there are many queries
    plt.xticks(fontsize = fontsize, rotation=0)
    plt.yticks(fontsize = fontsize)
    plt.tight_layout()
    plt.show()


# Read all CSV files from the different_errors directory
def load_experimental_data(directory):
    all_data = []
    
    # Create a Path object for the directory
    dir_path = Path(directory)
    
    # Iterate through all CSV files in the directory
    for file_path in dir_path.glob('*.csv'):
        df = pd.read_csv(file_path)
        df['run'] = int(file_path.stem.split('_')[-1].replace('run', ''))
        df['errorBound'] = float(file_path.stem.split('_')[1])
        all_data.append(df)
    
    # Combine all dataframes
    return pd.concat(all_data, ignore_index=True)

# Function to plot the aggregated results
def plot_aggregated_error_per_column(directory, y_col, include_initialization=False):
    # Load the data
    df = load_experimental_data(directory)

    # Group by errorBound and query number, then calculate mean and std
    agg_df = df.groupby(['errorBound', 'i']).agg(
        mean_time=(y_col, 'mean'),
        std_time=(y_col, 'std')
    ).reset_index()

    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]


    color_palette = {error_bound: colors[i % len(colors)] for i, error_bound in enumerate(agg_df['errorBound'].unique())}

    # Create query index for x-axis (assuming 'i' column represents query sequence)
    plt.figure(figsize=(12, 5))
    fontsize=14
    # Plot a line for each error bound without markers
    for error_bound in agg_df['errorBound'].unique():
        data = agg_df[agg_df['errorBound'] == error_bound]
        label = 'Exact' if error_bound == 0 else f'{error_bound * 100:.0f}%'
        plt.plot(data['i'], data['mean_time'], 
                 label=label, 
                 linewidth=2,
                 color=color_palette[error_bound])

    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel(y_col, fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize = fontsize, frameon=False)  # Remove the box around the legend
    
    # Remove the grid lines behind the plot
    plt.grid(False)

    # Rotate x-axis labels if there are many queries
    plt.xticks(fontsize = fontsize, rotation=0)
    plt.yticks(fontsize = fontsize)

    # Adjust x-axis to start from 1 if initialization is excluded
    if not include_initialization:
        plt.xlim(left=1)

    plt.tight_layout()
    plt.show()

# Function to plot the line chart along with a bar chart showing the summation of time compared to all errors
def plot_aggregated_error_per_column_with_bars(directory, y_col, include_initialization=False):
    # Load the data
    df = load_experimental_data(directory)

    # Group by errorBound and query number, then calculate mean and std
    agg_df = df.groupby(['errorBound', 'i']).agg(
        mean_time=(y_col, 'mean'),
        std_time=(y_col, 'std')
    ).reset_index()

    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]



    color_palette = {error_bound: colors[i % len(colors)] for i, error_bound in enumerate(agg_df['errorBound'].unique())}

    # Create a figure with two subplots side by side with different sizes
    fig = plt.figure(figsize=(15, 5))
    gs = fig.add_gridspec(1, 2, width_ratios=[3, 1])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])
    fontsize=14

    # Plot a line for each error bound without markers on the first subplot
    for error_bound in agg_df['errorBound'].unique():
        data = agg_df[agg_df['errorBound'] == error_bound]
        label = 'Exact' if error_bound == 0 else f'{error_bound * 100:.0f}%'
        ax1.plot(data['i'], data['mean_time'], 
                 label=label, 
                 linewidth=2,
                 color=color_palette[error_bound])

    ax1.set_xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    ax1.set_ylabel(y_col, fontsize=fontsize, fontweight='bold')
    ax1.legend(fontsize = fontsize, frameon=False)  # Remove the box around the legend
    
    # Remove the grid lines behind the plot
    ax1.grid(False)

    # Rotate x-axis labels if there are many queries
    ax1.tick_params(axis='x', labelsize=fontsize, rotation=0)
    ax1.tick_params(axis='y', labelsize=fontsize)

    # Adjust x-axis to start from 1 if initialization is excluded
    if not include_initialization:
        ax1.set_xlim(left=1)

    # Create the bar chart on the second subplot
    total_times = agg_df.groupby('errorBound')['mean_time'].sum().reset_index()
    total_times['label'] = total_times['errorBound'].apply(lambda x: 'Exact' if x == 0 else f'{x * 100:.0f}%')
    bars = ax2.bar(total_times['label'], total_times['mean_time'], alpha=0.7)

    # Set the colors of the bars to match the line colors
    for bar, error_bound in zip(bars, total_times['errorBound']):
        bar.set_color(color_palette[error_bound])

    ax2.set_xlabel('Error Bound', fontsize=fontsize, fontweight='bold')
    ax2.set_ylabel(f'Total {y_col}', fontsize=fontsize, fontweight='bold')
    ax2.tick_params(axis='x', labelsize=fontsize, rotation=0)
    ax2.tick_params(axis='y', labelsize=fontsize)

    plt.tight_layout()
    plt.show()

# Function to plot only the bar chart showing the summation of time compared to all errors
def plot_aggregated_error_per_column_bars_only(directory, y_col, include_initialization=False):
    # Load the data
    df = load_experimental_data(directory)

    # Group by errorBound and query number, then calculate mean and std
    agg_df = df.groupby(['errorBound', 'i']).agg(
        mean_time=(y_col, 'mean'),
        std_time=(y_col, 'std')
    ).reset_index()

    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]

    color_palette = {error_bound: colors[i % len(colors)] for i, error_bound in enumerate(agg_df['errorBound'].unique())}

    # Create query index for x-axis (assuming 'i' column represents query sequence)
    plt.figure(figsize=(12, 5))
    fontsize=14

    # Create the bar chart
    total_times = agg_df.groupby('errorBound')['mean_time'].sum().reset_index()
    total_times['label'] = total_times['errorBound'].apply(lambda x: 'Exact' if x == 0 else f'{x * 100:.0f}%')
    bars = plt.bar(total_times['label'], total_times['mean_time'], alpha=0.7)

    # Set the colors of the bars to match the line colors
    for bar, error_bound in zip(bars, total_times['errorBound']):
        bar.set_color(color_palette[error_bound])

    plt.xlabel('Error Bound', fontsize=fontsize, fontweight='bold')
    plt.ylabel(f'Total {y_col}', fontsize=fontsize, fontweight='bold')

    # Rotate x-axis labels if there are many queries
    plt.xticks(fontsize = fontsize, rotation=0)
    plt.yticks(fontsize = fontsize)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_aggregated_error_per_column(directory, 'Time (sec)')
plot_aggregated_error_per_column_with_bars(directory, 'Time (sec)', include_initialization=False)
plot_aggregated_error_per_column_bars_only(directory, 'Time (sec)', include_initialization=False)

In [ ]:
plot_aggregated_error_per_column(directory, 'I/Os')
plot_aggregated_error_per_column_with_bars(directory, 'I/Os', include_initialization=False)
plot_aggregated_error_per_column_bars_only(directory, 'I/Os', include_initialization=False)

In [ ]:
def compare_exact_vs_approx_results(directory, include_initialization=True, approx_error_bound=0.05):
    # Load data using existing function 
    df = load_experimental_data(directory)
    
     # Group by errorBound and query number, then calculate mean and std
    agg_df = df.groupby(['errorBound', 'i']).agg(
        conf_lb=('Confidence Interval LB', 'mean'),
        conf_ub=('Confidence Interval UB', 'mean'),
        error_bound=('Error Bound', 'mean'),
        val=('Query Result Sum', 'mean'),
    ).reset_index()
    # Optionally exclude the first point (initialization time)
    if not include_initialization:
        agg_df = agg_df[agg_df['i'] != 0]
        
    # Get exact and approximate results
    exact_df = agg_df[agg_df['errorBound'] == 0.0]
    approx_df = agg_df[agg_df['errorBound'] == approx_error_bound]
    
    # Initialize results storage
    results = []
    
    # For each query in the approximate results
    for i in range(len(approx_df)):
        approx_row = approx_df.iloc[i]
        
        # Find matching query in exact results
        exact_row = exact_df[exact_df['i'] == approx_row['i']]
        
        if len(exact_row) == 0:
            continue
            
        exact_sum = exact_row.iloc[0]['val']
        
        # Get confidence interval and error bound
        ci_lb = approx_row['conf_lb']
        ci_ub = approx_row['conf_ub']
        error_bound = approx_row['error_bound']
        
        # Check if exact sum falls within confidence interval
        within_ci = ci_lb <= exact_sum <= ci_ub
        if(not within_ci):
            print(f"Query {i} - Exact: {exact_sum}, CI: ({ci_lb}, {ci_ub})")
        # Calculate actual relative error 
        if exact_sum != 0:
            actual_error = abs(((ci_lb + ci_ub)/2 - exact_sum) / exact_sum)
        else:
            actual_error = float('inf')
            
        results.append({
            'query_index': i,
            'exact_sum': exact_sum,
            'estimated_sum': (ci_lb + ci_ub)/2,
            'ci_lb': ci_lb,
            'ci_ub': ci_ub, 
            'error_bound': error_bound,
            'actual_error': actual_error,
            'within_ci': within_ci
        })
    
    # Convert results to DataFrame
    results_df = pd.DataFrame(results)

    # Plot results
    plt.figure(figsize=(12, 5))
    fontsize = 14
    
    # Plot exact values
    plt.plot(results_df['query_index'], results_df['exact_sum'], label='Exact', linewidth=2, color='black')
    
    # Plot confidence intervals
    plt.fill_between(results_df['query_index'], 
                     results_df['ci_lb'],
                     results_df['ci_ub'],
                     alpha=0.2,
                     color='red',
                     label='Confidence Interval')
    
    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel('Query Result Sum', fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize=fontsize, frameon=False)
    plt.grid(False)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tight_layout()
    plt.show()
    
    # Calculate summary statistics
    summary = {
        'total_queries': len(results),
        'queries_within_ci': results_df['within_ci'].sum(),
        'avg_actual_error': results_df['actual_error'].mean(),
        'max_actual_error': results_df['actual_error'].max(),
        'avg_error_bound': results_df['error_bound'].mean()
    }
    
    return results_df, summary


def compare_all_error_bounds(directory):
    # Load all data using existing function
    df = load_experimental_data(directory)
    
    # Get unique error bounds and sort them
    error_bounds = sorted(df['errorBound'].unique())
    
    # Get exact results (error bound 0)
    exact_df = df[df['errorBound'] == 0]
    
    plt.figure(figsize=(12, 5))
    fontsize = 14
    
    # Plot exact values
    plt.plot(exact_df['i'], exact_df['Query Result Sum'], 
             label='Exact', linewidth=2, color='black')
    
    # Plot each error bound result
    colors = ['red', 'green', 'blue', 'orange', 'purple']  # Add more colors if needed
    for i, error_bound in enumerate([eb for eb in error_bounds if eb != 0]):
        approx_df = df[df['errorBound'] == error_bound]
        
        # Plot confidence intervals
        plt.fill_between(approx_df['i'], 
                        approx_df['Confidence Interval LB'],
                        approx_df['Confidence Interval UB'],
                        alpha=0.2,
                        color=colors[i % len(colors)],
                        label=f'{int(error_bound * 100)} %')
    
    plt.xlabel('Query Sequence', fontsize=fontsize, fontweight='bold')
    plt.ylabel('Query Result Sum', fontsize=fontsize, fontweight='bold')
    plt.legend(fontsize=fontsize, frameon=False)
    plt.grid(False)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    plt.tight_layout()
    plt.show()

    # Print summary statistics for each error bound
    for error_bound in error_bounds:
        bound_df = df[df['errorBound'] == error_bound]
        if error_bound == 0:
            continue
            
        exact_values = exact_df['Query Result Sum'].values
        estimated_values = (bound_df['Confidence Interval LB'] + bound_df['Confidence Interval UB'])/2
        actual_errors = np.abs(estimated_values - exact_values)/exact_values
        
        within_bounds = ((bound_df['Confidence Interval LB'] <= exact_values) & 
                        (bound_df['Confidence Interval UB'] >= exact_values)).mean()
        
        print(f"\nError Bound {error_bound}:")
        print(f"Average Actual Error: {actual_errors.mean():.4f}")
        print(f"Max Actual Error: {actual_errors.max():.4f}")
        print(f"Queries within bounds: {within_bounds*100:.1f}%")


In [ ]:
compare_exact_vs_approx_results(directory, approx_error_bound=0.05)

In [ ]:
compare_all_error_bounds(directory)
